### RAG Pipelines - Data Ingestion to Vector DB pipeline

In [9]:
import os 
from langchain_community.document_loaders import PyMuPDFLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [ ]:
def process_all_pdfs(pdf_dir):
    """Process all PDF files in the specified directory and return a list of documents with metadata."""
    all_documents = []
    pdf_dir_path = Path(pdf_dir)

    pdf_files = list(pdf_dir_path.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process.")

    for pdf_file in pdf_files:
        print(f"\nProcessing {pdf_file}...")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source metadata to each document, additional metadata can be added here as needed
            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = 'pdf'

            all_documents.extend(documents)
            print(f"\nLoaded {len(documents)} documents from {pdf_file}.")

        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

all_docs = process_all_pdfs("../data/pdf")
print(f"\nTotal documents processed: {len(all_docs)}")

In [22]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks using RecursiveCharacterTextSplitter for RAG processing."""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks.")

    if split_docs:
        print(f"\nSample split document metadata: {split_docs[0].metadata}")
        print(f"\nSample split document content (first 200 chars): {split_docs[0].page_content[:200]}")
    return split_docs

In [ ]:
chunked_docs = split_documents(all_docs)
print(f"\nTotal chunked documents: {len(chunked_docs)}")

### Embedding and VectorstoreDB

In [29]:
import numpy as np 
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [30]:
class EmbeddingManager:
    """Manages embedding generation using SentenceTransformer and storage/retrieval in ChromaDB."""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager.
        
        Args:
            model_name (str): The name of the HuggingFace model to use for embeddings.
            """
        
        self.model_name = model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.
        
        Args:
            texts (List[str]): A list of strings to generate embeddings for.
        Returns:
            np.ndarray: An array of embeddings corresponding to the input texts.   
        """

        if not self.model:
            raise ValueError("Embedding model is not loaded.") 

# Initialize the embedding manager and generate embeddings for the chunked documents
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7185.85it/s]


Model loaded successfully. Embedding dimension: 384
